# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aanvi02/ML-work/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

I will use a transparent, hand-written rule to prioritize content for human review. This is a decision-support baseline, not a prediction of guaranteed traffic improvement.

The rule will use only existing, observable content and performance measurements. It will not use `trend_direction`, `trend_pct`, existing product flags, or future-window information as scoring inputs.

### Reason codes

- **STALE_REVIEW** — content has not been updated recently and should be reviewed for freshness.
- **CTR_REVIEW** — content receives impressions but its observed CTR may warrant a closer look, considering its average search position.
- **VOLUME_REVIEW** — content has meaningful search demand and may be worth reviewing for a quick-win opportunity.
- **GENERAL_REVIEW** — fallback reason when the row does not meet the selected conditions.

Each item will receive one primary reason code, one action label, a transparent numeric score, and a queue rank. The top 20 items will be reviewed manually, including confidence notes and reasons each recommendation could be wrong.

### Limitations

These signals are observational. They do not prove that refreshing a page will improve traffic. Missing values, low sample sizes, seasonality, and differences between content topics may affect the recommendations.


In [2]:
from pathlib import Path

print("Current folder:", Path.cwd())
print("Files here:", [p.name for p in Path.cwd().iterdir()])


Current folder: /content
Files here: ['.config', 'sample_data']


In [3]:
!git clone https://github.com/Aanvi02/ML-work.git /content/ML-work
!ls -R /content/ML-work/data

Cloning into '/content/ML-work'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (106/106), done.
remote: Total 149 (delta 59), reused 91 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 1.87 MiB | 13.05 MiB/s, done.
Resolving deltas: 100% (59/59), done.
/content/ML-work/data:
raw

/content/ML-work/data/raw:
content_refresh_anonymized.csv


In [4]:
from pathlib import Path
import pandas as pd
import numpy as np

repo = Path("/content/ML-work")
data_path = repo / "data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types and missing values:")
display(df.info())

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

First 5 rows:


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7



Data types and missing values:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pagevi

None

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:

# ML-07 — Baseline Action Score
from pathlib import Path
import numpy as np
import pandas as pd

# Load data directly so the notebook can run from the repo.
repo = Path("/content/ML-work")
data_path = repo / "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

scored = df.copy()

# Conservative, transparent thresholds.
# Missing search volume is not treated as high demand.
scored["stale_signal"] = scored["days_since_last_update"] >= 180
scored["volume_signal"] = (
    scored["search_volume"].notna()
    & (scored["search_volume"] >= scored["search_volume"].quantile(0.75))
)

# CTR review: compare observed CTR with the median CTR for
# the same broad position band, and require at least 100 impressions.
scored["position_bucket"] = pd.cut(
    scored["avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=["1-3", "3-5", "5-10", "10-20", "20+"],
    include_lowest=True
)

position_ctr_median = scored.groupby(
    "position_bucket", observed=False
)["ctr"].transform("median")

scored["ctr_signal"] = (
    (scored["impressions_90d"] >= 100)
    & (scored["ctr"] < position_ctr_median)
)

# Exactly one primary reason code per row.
# Scores are additive and fully hand-written.
scored["score"] = (
    scored["stale_signal"].astype(int) * 2
    + scored["ctr_signal"].astype(int) * 2
    + scored["volume_signal"].astype(int)
)

scored["reason_code"] = np.select(
    [
        scored["stale_signal"] & scored["ctr_signal"],
        scored["stale_signal"],
        scored["ctr_signal"],
        scored["volume_signal"]
    ],
    [
        "STALE_REVIEW",
        "STALE_REVIEW",
        "CTR_REVIEW",
        "VOLUME_REVIEW"
    ],
    default="GENERAL_REVIEW"
)

scored["action_label"] = np.select(
    [
        scored["reason_code"].eq("STALE_REVIEW"),
        scored["reason_code"].eq("CTR_REVIEW"),
        scored["reason_code"].eq("VOLUME_REVIEW")
    ],
    [
        "Review freshness and accuracy",
        "Review title, snippet, and search intent",
        "Review quick-win optimization opportunity"
    ],
    default="Manual content review"
)

# Stable ranking: score first, then impressions, then content ID.
scored = scored.sort_values(
    ["score", "impressions_90d", "content_id"],
    ascending=[False, False, True],
    kind="mergesort"
).reset_index(drop=True)

scored["rank_queue"] = np.arange(1, len(scored) + 1)

# Output only necessary fields; avoid exporting client identifiers.
output_columns = [
    "content_id",
    "score",
    "reason_code",
    "action_label",
    "rank_queue",
    "days_since_last_update",
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position"
]

out_dir = repo / "work/outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "baseline_action_score.csv"

scored[output_columns].to_csv(out_path, index=False)

print("Rows scored:", len(scored))
print("Output saved to:", out_path)
print("\nReason-code counts:")
display(scored["reason_code"].value_counts().rename_axis("reason_code").to_frame("n"))
print("\nTop 10 ranked items:")
display(scored[output_columns].head(10))


Rows scored: 30000
Output saved to: /content/ML-work/work/outputs/baseline_action_score.csv

Reason-code counts:


,n
reason_code,
GENERAL_REVIEW,17150
VOLUME_REVIEW,7285
CTR_REVIEW,5391
STALE_REVIEW,174



Top 10 ranked items:


,content_id,score,reason_code,action_label,rank_queue,days_since_last_update,search_volume,impressions_90d,clicks_90d,ctr,avg_position
0,content_02b0d6e30129,5,STALE_REVIEW,Review freshness and accuracy,1,313,110.0,176,0,0.00,6.9
1,content_fd16e3475c29,4,STALE_REVIEW,Review freshness and accuracy,2,183,0.0,429,0,0.00,9.0
2,content_4f241bad48a3,4,STALE_REVIEW,Review freshness and accuracy,3,236,10.0,285,0,0.00,19.1
3,content_ea41fe5cf292,4,STALE_REVIEW,Review freshness and accuracy,4,183,0.0,265,0,0.00,8.0
4,content_958a46db26bd,4,STALE_REVIEW,Review freshness and accuracy,5,183,0.0,198,0,0.00,6.7
5,content_f488400fca67,4,STALE_REVIEW,Review freshness and accuracy,6,305,NaN,155,0,0.00,5.7
6,content_b6e4581523ed,4,STALE_REVIEW,Review freshness and accuracy,7,183,0.0,104,0,0.00,14.0
7,content_ab27c30d81f4,4,STALE_REVIEW,Review freshness and accuracy,8,304,NaN,103,0,0.00,8.9
8,content_5fe46e04994d,3,CTR_REVIEW,"Review title, snippet, and search intent",9,104,1900.0,517715,741,0.14,4.2
9,content_c8e9d6ab9013,3,CTR_REVIEW,"Review title, snippet, and search intent",10,104,20.0,208678,0,0.00,9.7


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [6]:

# ML-07 — Human review of the top 20
top20 = scored.head(20).copy()

def confidence_note(row):
    if row["impressions_90d"] < 100:
        return "Low: limited impression volume"
    if pd.isna(row["search_volume"]):
        return "Moderate-low: search volume is missing"
    return "Moderate: based on observed metrics; validate manually"

def could_be_wrong(row):
    if row["reason_code"] == "STALE_REVIEW":
        return "The page may still be accurate; update date alone does not prove decay."
    if row["reason_code"] == "CTR_REVIEW":
        return "CTR may be affected by query mix, SERP features, or position differences."
    if row["reason_code"] == "VOLUME_REVIEW":
        return "Search demand may be competitive or poorly matched to this page."
    return "No strong rule signal; manual context may change the priority."

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(could_be_wrong, axis=1)

review_columns = [
    "rank_queue",
    "content_id",
    "action_label",
    "reason_code",
    "score",
    "confidence_note",
    "what_would_make_it_wrong"
]

display(top20[review_columns])

print("Top-20 review rows:", len(top20))
assert len(top20) == min(20, len(scored))


,rank_queue,content_id,action_label,reason_code,score,confidence_note,what_would_make_it_wrong
0,1,content_02b0d6e30129,Review freshness and accuracy,STALE_REVIEW,5,Moderate: based on observed metrics; validate ...,The page may still be accurate; update date al...
1,2,content_fd16e3475c29,Review freshness and accuracy,STALE_REVIEW,4,Moderate: based on observed metrics; validate ...,The page may still be accurate; update date al...
2,3,content_4f241bad48a3,Review freshness and accuracy,STALE_REVIEW,4,Moderate: based on observed metrics; validate ...,The page may still be accurate; update date al...
3,4,content_ea41fe5cf292,Review freshness and accuracy,STALE_REVIEW,4,Moderate: based on observed metrics; validate ...,The page may still be accurate; update date al...
4,5,content_958a46db26bd,Review freshness and accuracy,STALE_REVIEW,4,Moderate: based on observed metrics; validate ...,The page may still be accurate; update date al...
5,6,content_f488400fca67,Review freshness and accuracy,STALE_REVIEW,4,Moderate-low: search volume is missing,The page may still be accurate; update date al...
6,7,content_b6e4581523ed,Review freshness and accuracy,STALE_REVIEW,4,Moderate: based on observed metrics; validate ...,The page may still be accurate; update date al...
7,8,content_ab27c30d81f4,Review freshness and accuracy,STALE_REVIEW,4,Moderate-low: search volume is missing,The page may still be accurate; update date al...
8,9,content_5fe46e04994d,"Review title, snippet, and search intent",CTR_REVIEW,3,Moderate: based on observed metrics; validate ...,"CTR may be affected by query mix, SERP feature..."
9,10,content_c8e9d6ab9013,"Review title, snippet, and search intent",CTR_REVIEW,3,Moderate: based on observed metrics; validate ...,"CTR may be affected by query mix, SERP feature..."


Top-20 review rows: 20


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks and limitations

The ranked queue is a first-pass review list, not a claim that these pages will improve after an update.

Potentially weak picks include:
- Old pages that remain accurate and useful despite not being updated recently.
- CTR-review items whose click-through rate is explained by query mix, search features, or position.
- High-volume items that are highly competitive or do not match the intended search intent.
- Rows with missing search-volume data or limited impressions.

### Leakage check

The scoring rule does not use `trend_direction`, `trend_pct`, or existing product flags. It uses current dataset measurements and does not construct features from future performance windows. The score is hand-written rather than trained on a future outcome label.

The output is decision support only. The top 20 require human review before any content change is recommended.

In [7]:

# ML-07 — Weak-pick and leakage checks

# Verify output and ranking integrity.
assert len(scored) == len(df)
assert scored["rank_queue"].is_unique
assert scored["rank_queue"].min() == 1
assert scored["rank_queue"].max() == len(scored)
assert scored["reason_code"].notna().all()
assert scored["action_label"].notna().all()

# Confirm forbidden fields are not used in the scoring expression.
forbidden_scoring_inputs = {"trend_direction", "trend_pct"}
actual_scoring_inputs = {
    "days_since_last_update",
    "search_volume",
    "impressions_90d",
    "ctr",
    "avg_position"
}
assert forbidden_scoring_inputs.isdisjoint(actual_scoring_inputs)

# Flag potentially weak top-20 picks for human review.
weak_picks = top20[
    (top20["impressions_90d"] < 100)
    | (top20["search_volume"].isna())
].copy()

print("Potential weak picks in top 20:", len(weak_picks))
display(
    weak_picks[
        [
            "rank_queue",
            "content_id",
            "reason_code",
            "impressions_90d",
            "search_volume",
            "confidence_note",
            "what_would_make_it_wrong"
        ]
    ]
)

print("\nLeakage checks passed.")
print("CSV exists:", out_path.exists())
print("CSV rows:", pd.read_csv(out_path).shape[0])


Potential weak picks in top 20: 2


,rank_queue,content_id,reason_code,impressions_90d,search_volume,confidence_note,what_would_make_it_wrong
5,6,content_f488400fca67,STALE_REVIEW,155,NaN,Moderate-low: search volume is missing,The page may still be accurate; update date al...
7,8,content_ab27c30d81f4,STALE_REVIEW,103,NaN,Moderate-low: search volume is missing,The page may still be accurate; update date al...



Leakage checks passed.
CSV exists: True
CSV rows: 30000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.